In [1]:
import json
import requests
from bs4 import BeautifulSoup

def extract_claim_and_evidence_from_politifact(tsv_line, header):
    fields = tsv_line.strip().split("\t")
    if len(fields) < 3:
        return None  # Skip malformed lines

    row = dict(zip(header, fields))
    
    claim = row["statement"]
    label = row["label"]
    id_raw = row["ID"].replace(".json", "")
    url = f"https://www.politifact.com/factchecks/{id_raw}/"

    evidence_text = []

    if url:
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Find the <article> tag which contains the main fact-check content
            article = soup.find("article", class_="m-textblock")
            if article:
                paragraphs = article.find_all("p")
                evidence_text = [p.get_text().strip() for p in paragraphs if p.get_text().strip()]
            else:
                print(f"No <article> tag found for {url}")
        except Exception as e:
            print(f"Error fetching {url}: {e}")

    return {
        "claim": claim,
        "label": label,
        "evidence_text": evidence_text
    }

In [4]:
results = []
with open("./LIAR_train.tsv", "r", encoding="utf-8") as f:
    header_line = f.readline().strip()
    header = header_line.split("\t")

    for i, line in enumerate(f):
        result = extract_claim_and_evidence_from_politifact(line, header)
        if result and result["evidence_text"]:  # Only keep if evidence exists
            results.append(result)
        if i % 100 == 0:
            print("at", i)
        if i == 5000:
            break

with open("LIAR_extracted.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

at 0
at 100
at 200
No <article> tag found for https://www.politifact.com/factchecks/13370/
at 300
at 400
at 500
at 600
at 700
at 800
No <article> tag found for https://www.politifact.com/factchecks/13346/
at 900
at 1000
at 1100
at 1200
at 1300
No <article> tag found for https://www.politifact.com/factchecks/12591/
at 1400
at 1500
at 1600
at 1700
No <article> tag found for https://www.politifact.com/factchecks/13094/
at 1800
at 1900
at 2000
at 2100
at 2200
at 2300
at 2400
at 2500
at 2600
at 2700
at 2800
at 2900
at 3000
at 3100
at 3200
at 3300
at 3400
No <article> tag found for https://www.politifact.com/factchecks/13148/
at 3500
at 3600
at 3700
at 3800
at 3900
No <article> tag found for https://www.politifact.com/factchecks/13412/
at 4000
at 4100
at 4200
at 4300
at 4400
at 4500
at 4600
at 4700
at 4800
at 4900
at 5000
